# Aesthetic AI — Kaggle Training Notebook

**Prerequisites:**
- Add your `aesthetic-pairs-queue` Kaggle Dataset as input (contains `queue.db` or `merged_queue.db`)
- Enable GPU accelerator: T4 x1
- Run cells **one at a time** and verify each passes before proceeding

In [ ]:
# Cell 1: Clone repo + install deps

GITHUB_REPO = "https://github.com/krutckwang/aesthetic-ai.git"
REPO_DIR = "/kaggle/working/aesthetic-ai"

import os, subprocess, sys, numpy as np

# Patch PIL._util in memory BEFORE importing torchvision.
import PIL._util as _pil_util
if not hasattr(_pil_util, 'is_directory'):
    import os as _os
    from pathlib import Path as _Path
    _pil_util.is_directory = lambda fp: _os.path.isdir(fp)
    _pil_util.is_path = lambda f: isinstance(f, (bytes, str, _Path))

import torch, torchvision
from PIL import Image as _PIL

with open('/tmp/kaggle_constraints.txt', 'w') as f:
    f.write(f"torch=={torch.__version__}\n")
    f.write(f"torchvision=={torchvision.__version__}\n")
    f.write(f"Pillow=={_PIL.__version__}\n")
    f.write(f"numpy=={np.__version__}\n")

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
else:
    # Use fetch+reset instead of pull to avoid branch tracking issues
    os.system(f'git -C {REPO_DIR} fetch origin main')
    os.system(f'git -C {REPO_DIR} reset --hard origin/main')
%cd {REPO_DIR}

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-c', '/tmp/kaggle_constraints.txt',
    '--upgrade-strategy', 'only-if-needed',
    'diffusers>=0.27.0', 'peft>=0.11.0', 'accelerate>=0.30.0',
    'insightface>=0.7.3', 'onnxruntime>=1.18.0', 'mediapipe>=0.10.14',
    'sqlalchemy>=2.0.30', 'alembic>=1.13.1',
    'loguru', 'tqdm', 'pyyaml', 'python-dotenv',
], check=True)

print(f"torch:       {torch.__version__}   (must contain +cu128)")
print(f"torchvision: {torchvision.__version__}")
print(f"Pillow:      {_PIL.__version__}")
print(f"numpy:       {np.__version__}")

In [ ]:
# Cell 2: Verify environment
# Expected: CUDA=True, VRAM >= 15 GB. Raise early if GPU not attached.

import torch, torchvision
from PIL import Image

print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Pillow:      {Image.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:         {props.name}")
    print(f"VRAM:        {props.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected — enable T4 accelerator in Notebook settings")

In [ ]:
# Cell 3: Download images from queue.db
# Accepts merged_queue.db (from scripts/merge_queue.py) or original queue.db.

import sqlite3, httpx, json, os, sys, shutil, time, random
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '/kaggle/working/aesthetic-ai')

# Try merged queue first, then original — supports both upload workflows
QUEUE_DB_CANDIDATES = [
    "/kaggle/input/aesthetic-pairs-queue/merged_queue.db",
    "/kaggle/input/aesthetic-pairs-queue/queue.db",
]
QUEUE_DB_SRC = next((p for p in QUEUE_DB_CANDIDATES if os.path.exists(p)), None)
if QUEUE_DB_SRC is None:
    raise FileNotFoundError(
        "Cannot find queue.db or merged_queue.db in Kaggle input.\n"
        "Add your 'aesthetic-pairs-queue' dataset as notebook input."
    )
print(f"Queue source: {QUEUE_DB_SRC}")
shutil.copy2(QUEUE_DB_SRC, '/kaggle/working/queue.db')

conn = sqlite3.connect('/kaggle/working/queue.db')
rows = conn.execute(
    "SELECT id, before_url, after_url, source_name, metadata "
    "FROM staging_queue WHERE status='pending'"
).fetchall()

# Show label coverage in the queue
labeled_in_queue = sum(
    1 for _, _, _, _, m in rows
    if m and 'treatment_category' in m
)
print(f"Queue pairs:     {len(rows)}")
print(f"Pre-labeled:     {labeled_in_queue} ({100*labeled_in_queue//max(len(rows),1)}%)")
conn.close()

IMAGE_DIR = Path('/kaggle/working/aesthetic-ai/data/raw')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

existing = sum(
    1 for r in rows
    if (IMAGE_DIR / f"{r[0]}_before.jpg").exists()
    and (IMAGE_DIR / f"{r[0]}_after.jpg").exists()
)
print(f"Already downloaded: {existing} — skipping those")

downloaded, skipped = 0, 0
with httpx.Client(
    timeout=30, follow_redirects=True, verify=False,
    headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
) as client:
    for row_id, before_url, after_url, source_name, metadata_str in tqdm(rows):
        b_path = IMAGE_DIR / f"{row_id}_before.jpg"
        a_path = IMAGE_DIR / f"{row_id}_after.jpg"
        try:
            if not b_path.exists():
                resp = client.get(before_url); resp.raise_for_status()
                b_path.write_bytes(resp.content)
            if not a_path.exists():
                resp = client.get(after_url); resp.raise_for_status()
                a_path.write_bytes(resp.content)
            downloaded += 1
            time.sleep(random.uniform(0.3, 0.8))
        except Exception:
            skipped += 1

print(f"Downloaded: {downloaded}  Failed/skipped: {skipped}")

In [ ]:
# Cell 4: Build manifest.json with face-detection filter + treatment label mapping

import subprocess, sys, sqlite3, json, torch
from pathlib import Path
from PIL import Image as PILImage
from tqdm import tqdm

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'facenet-pytorch'],
    check=True
)

from facenet_pytorch import MTCNN

IMAGE_DIR     = Path('/kaggle/working/aesthetic-ai/data/raw')
MANIFEST_PATH = Path('/kaggle/working/aesthetic-ai/data/manifest.json')
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

SLUG_MAP = {
    # Botox / neurotoxins
    'botulinum-toxin':                'botox',
    'botox':                          'botox',
    'botox-cosmetic':                 'botox',
    'dysport':                        'botox',
    'xeomin':                         'botox',
    'jeuveau':                        'botox',
    'daxi':                           'botox',
    # Lip filler
    'lip-augmentation':               'lip_filler',
    'lip-augmentation---enhancement': 'lip_filler',
    'lip-enhancement':                'lip_filler',
    'lip-filler':                     'lip_filler',
    'lip-fillers':                    'lip_filler',
    # Dermal fillers
    'dermal-fillers':                 'dermal_filler',
    'dermal-filler':                  'dermal_filler',
    'fillers':                        'dermal_filler',
    'juvederm':                       'dermal_filler',
    'restylane':                      'dermal_filler',
    'sculptra':                       'dermal_filler',
    'radiesse':                       'dermal_filler',
    'belotero':                       'dermal_filler',
    'cheek-augmentation':             'dermal_filler',
    'cheek-filler':                   'dermal_filler',
    'cheek-fillers':                  'dermal_filler',
    # Jawline / chin
    'chin-augmentation':              'jawline_filler',
    'chin-filler':                    'jawline_filler',
    'chin-implants':                  'jawline_filler',
    'jawline-filler':                 'jawline_filler',
    # Under-eye
    'under-eye-filler':               'under_eye_filler',
    'tear-trough':                    'under_eye_filler',
    # Kybella
    'kybella':                        'kybella',
    # Surgical
    'facelift':                       'facelift',
    'face-lift':                      'facelift',
    'mini-facelift':                  'facelift',
    'brow-lift':                      'facelift',
    'browlift':                       'facelift',
    'neck-lift':                      'facelift',
    'necklift':                       'facelift',
    'eyelid-surgery':                 'blepharoplasty',
    'blepharoplasty':                 'blepharoplasty',
    'upper-blepharoplasty':           'blepharoplasty',
    'lower-blepharoplasty':           'blepharoplasty',
    'upper-eyelid-surgery':           'blepharoplasty',
    'lower-eyelid-surgery':           'blepharoplasty',
    'rhinoplasty':                    'rhinoplasty',
    'nose-surgery':                   'rhinoplasty',
    'nose-reshaping':                 'rhinoplasty',
    'otoplasty':                      'otoplasty',
    'ear-surgery':                    'otoplasty',
    'fat-transfer-to-face':           'fat_transfer',
    'fat-transfer':                   'fat_transfer',
    # Skin treatments
    'laser-skin-resurfacing':         'laser_resurfacing',
    'laser-resurfacing':              'laser_resurfacing',
    'chemical-peel':                  'chemical_peel',
    'chemical-peels':                 'chemical_peel',
    'microneedling':                  'microneedling',
    'thread-lift':                    'thread_lift',
    'microdermabrasion':              'microdermabrasion',
    'ipl-photofacial':                'ipl_photofacial',
    'ipl':                            'ipl_photofacial',
    'prp':                            'prp',
}

# Subreddit name → treatment (Reddit source label)
SUBREDDIT_MAP = {
    'rhinoplasty':          'rhinoplasty',
    'jawsurgery':           'jawline_filler',
    'facialplasticsurgery': 'facelift',
    'eyelidsurgery':        'blepharoplasty',
    'fillers':              'dermal_filler',
    'injectables':          'dermal_filler',
    'botox':                'botox',
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mtcnn  = MTCNN(keep_all=False, device=device, min_face_size=40)

conn = sqlite3.connect('/kaggle/working/queue.db')
rows = conn.execute(
    "SELECT id, before_url, after_url, source_url, metadata "
    "FROM staging_queue WHERE status='pending'"
).fetchall()
conn.close()

def extract_treatment(source_url, metadata_str):
    # 1. Prefer treatment stored in metadata by the crawler
    try:
        meta = json.loads(metadata_str or '{}')
        if meta.get('treatment_category'):
            return meta['treatment_category']
    except Exception:
        pass
    # 2. Search all URL path segments
    if source_url:
        url = source_url.lower()
        # Reddit: extract subreddit name
        import re
        m = re.search(r'/r/([^/]+)', url)
        if m:
            sub = m.group(1).lower()
            if sub in SUBREDDIT_MAP:
                return SUBREDDIT_MAP[sub]
        # Generic slug matching
        for part in url.rstrip('/').split('/'):
            part = part.split('?')[0]
            if part in SLUG_MAP:
                return SLUG_MAP[part]
    return None

def has_face(path):
    try:
        boxes, _ = mtcnn.detect(PILImage.open(path).convert('RGB'))
        return boxes is not None and len(boxes) > 0
    except Exception:
        return False

records, no_image, no_face = [], 0, 0

for row_id, before_url, after_url, source_url, metadata_str in tqdm(rows, desc="face-detect"):
    b_path = IMAGE_DIR / f"{row_id}_before.jpg"
    a_path = IMAGE_DIR / f"{row_id}_after.jpg"
    if not (b_path.exists() and a_path.exists()):
        no_image += 1
        continue
    if not (has_face(b_path) and has_face(a_path)):
        no_face += 1
        continue

    treatment = extract_treatment(source_url, metadata_str)

    records.append({
        "pair_id":            row_id,
        "before_path":        str(b_path),
        "after_path":         str(a_path),
        "treatment_category": treatment,
        "treatment_brand":    None,
        "zone_codes":         [],
    })

MANIFEST_PATH.write_text(json.dumps(records, indent=2), encoding="utf-8")

labeled = sum(1 for r in records if r["treatment_category"])
print(f"Checked:           {len(rows)}")
print(f"Missing images:    {no_image}")
print(f"No face (skipped): {no_face}")
print(f"Manifest pairs:    {len(records)}")
print(f"Label coverage:    {labeled}/{len(records)} ({100*labeled//max(len(records),1)}%)")

# Label breakdown
from collections import Counter
counts = Counter(r['treatment_category'] for r in records if r['treatment_category'])
for treatment, n in counts.most_common():
    print(f"  {treatment:<25} {n}")

if len(records) < 200:
    raise RuntimeError(f"Only {len(records)} face pairs — too few to train.")

In [ ]:
# Cell 5: Train InstructPix2Pix + LoRA
# Run as a module (-m) so the project root is on sys.path.

import os
os.chdir('/kaggle/working/aesthetic-ai')
!git fetch origin main
!git reset --hard origin/main

!python -m model.training.train \
    --manifest        data/manifest.json \
    --base_model      timbrooks/instruct-pix2pix \
    --output_dir      /kaggle/working/lora_output \
    --num_steps       25000 \
    --batch_size      2 \
    --mixed_precision fp16 \
    --lora_rank       16 \
    --save_every      500 \
    --num_workers     0

In [ ]:
# Cell 6: Visual inference — before / predicted / actual
# Run after Cell 5 completes. Shows what the model predicts for sample pairs.

import os, json, random, sys, torch
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

REPO_DIR = '/kaggle/working/aesthetic-ai'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/krutckwang/aesthetic-ai.git {REPO_DIR}')
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from diffusers import StableDiffusionInstructPix2PixPipeline
from model.instruct_pix2pix.lora import LoRAConfig, inject_lora

# Find checkpoint (final or latest step)
lora_dir = Path('/kaggle/working/lora_output')
final    = lora_dir / 'final' / 'lora_weights.pt'
if not final.exists():
    candidates = sorted(lora_dir.glob('step_*/lora_weights.pt'))
    final = candidates[-1] if candidates else None
if final is None:
    raise FileNotFoundError('No LoRA checkpoint found — run Cell 5 first')
print(f"Loading: {final}")

# Load base pipeline
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    'timbrooks/instruct-pix2pix',
    torch_dtype=torch.float16,
    safety_checker=None,
).to('cuda')

# Inject LoRA and load weights
inject_lora(pipe.unet, LoRAConfig(rank=16))
lora_state = torch.load(final, map_location='cuda')
pipe.unet.load_state_dict(lora_state, strict=False)
pipe.unet.eval()
print(f"LoRA tensors loaded: {len(lora_state)}")

PROMPTS = {
    'botox':          'apply botox treatment, smooth forehead wrinkles',
    'lip_filler':     'apply lip filler, fuller lips',
    'dermal_filler':  'apply dermal filler, restore facial volume',
    'rhinoplasty':    'rhinoplasty result, refined nose shape',
    'facelift':       'facelift result, lifted and tightened facial skin',
    'blepharoplasty': 'blepharoplasty result, refreshed eye area',
    'jawline_filler': 'apply jawline filler, defined jaw',
    'kybella':        'kybella treatment, reduced submental fat',
    'chemical_peel':  'chemical peel result, smoother skin texture',
    'microneedling':  'microneedling result, improved skin texture',
}

records  = json.loads(Path('/kaggle/working/aesthetic-ai/data/manifest.json').read_text())
labeled  = [r for r in records if r.get('treatment_category')]
pool     = labeled if labeled else records
N        = min(4, len(pool))
samples  = random.sample(pool, N)

fig, axes = plt.subplots(N, 3, figsize=(13, 4 * N))
if N == 1:
    axes = [axes]

for i, rec in enumerate(samples):
    treatment = rec.get('treatment_category')
    prompt    = PROMPTS.get(treatment, 'apply aesthetic facial treatment')
    before    = Image.open(rec['before_path']).convert('RGB').resize((512, 512))
    after     = Image.open(rec['after_path']).convert('RGB').resize((512, 512))

    with torch.inference_mode():
        predicted = pipe(
            prompt=prompt,
            image=before,
            num_inference_steps=50,
            image_guidance_scale=1.5,
            guidance_scale=7.5,
        ).images[0]

    for ax, img, title in zip(
        axes[i],
        [before, predicted, after],
        ['Before', f'Predicted\n({treatment})', 'Actual After']
    ):
        ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')

plt.tight_layout()
out_path = '/kaggle/working/inference_results.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")
print("Download from Output tab → /kaggle/working/inference_results.png")